In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
import ezdxf
from rasterio.plot import show
import numpy as np

## 0.6 - Merge RegBL and Soils

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `12_SWISSTOPO_unconsolidated_soils`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Soils rasters

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/12_SWISSTOPO_unconsolidated_soils/TMUD25-2021.tif')

uncsoil_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
print("Min:", uncsoil_data.min())
print("Max:", uncsoil_data.max())

In [ ]:
# Mask the background (value = -999999999)
uncsoil_data = np.ma.masked_where(uncsoil_data == -999999999, uncsoil_data)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(uncsoil_data, cmap='viridis')
plt.show()

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.3.csv', index_col=0)

df_regbl.head()

## Convert current dataset to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Overlay soils raster on RegBL to obtain soil class for each building

In [ ]:
with rasterio.open(f'{ROOT}data/raw/12_SWISSTOPO_unconsolidated_soils/TMUD25-2021.tif') as src:
    soil_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    soil_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = soil_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        soil_values.append(val)

df_regbl["UncSoil - Classe de sol"] = soil_values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='UncSoil - Classe de sol', ax=ax, legend=True, cmap='viridis')
plt.show()

## Convert current dataset back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl, labels=True, fontsize=8)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.4.csv")